# <font color='red'>
---
## <center> <font color='red'> Masters in Mathematical Finance
#### <center> <font color='red'> 2024 / 26
# <center> <font color='red'> Masters' Final Work
---
# <center> <font color='red'><font> Student: Petr Terletskiy </font>
### <center> <font color='red'><font> Number: l63023 </font>
---
##### <center>  <font color='red'><font> BTC Daily Direction Prediction Using KANs Within the AFML Framework</font>

---

This notebook implements the full experimental pipeline for a Mathematical Finance Masters thesis at ISEG. 

The objective is to evaluate whether Kolmogorov–Arnold Networks (KANs) can predict Bitcoin daily price direction, benchmarked against AR Logistic, Logistic Regression, Random Forest, XGBoost, and LSTM models. 

Evaluation follows López de Prado's *Advances in Financial Machine Learning* (2018) framework: triple-barrier labeling, CUSUM event filtering, sample uniqueness weighting, and Combinatorial Purged Cross-Validation (CPCV, N=6, k=2) with purging and embargo to prevent information leakage. Post-CPCV analysis includes the Deflated Sharpe Ratio, Probability of Backtest Overfitting, and KAN symbolic extraction following the KASPER paper's Algorithm 1. Data covers BTC-USD daily OHLCV from October 2014 to March 2026, with 55 features (23 technical, 9 mathematical, 23 external including macro, crypto, and on-chain).

**Code structure:**
- `src/pre_cpcv/` (data loading, labeling, sample weights, feature engineering, alignment)
- `src/cpcv/` (cross-validation splits, preprocessing, models, calibration, pipeline orchestration)
- `src/post_cpcv/` (evaluation, symbolic extraction)

# 1. Dependencies

## 1.1. Installations

In [ ]:
%pip install yfinance coinmetrics.api_client matplotlib numpy pandas pandas-datareader statsmodels scikit-learn pyarrow optuna xgboost torch pykan pyyaml --quiet 
%pip install git+https://github.com/Blealtan/efficient-kan.git --quiet

## 1.2. Libraries & Project Modules

In [ ]:
%load_ext autoreload
%autoreload 2

# ── stdlib & third-party ──────────────────────────────────────────────
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from statsmodels.tsa.stattools import adfuller
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, roc_auc_score,
                             ConfusionMatrixDisplay, RocCurveDisplay,)
import sympy

# ── project modules ───────────────────────────────────────────────────
# Pre-CPCV
from src.pre_cpcv.data_loader import load_btc_daily
from src.pre_cpcv.labeling import compute_daily_volatility, cusum_filter, triple_barrier_labels, drop_rare_labels, run_labeling_pipeline
from src.pre_cpcv.sample_weights import compute_sample_weights
from src.pre_cpcv.features import compute_ta_features, compute_math_features, apply_log_transforms, build_feature_matrix
from src.pre_cpcv.external_features import build_external_features
from src.pre_cpcv.alignment import align_for_cv, validate_alignment

# CPCV
from src.cpcv.cv import generate_cpcv_splits, build_path_matrix, get_split_info
from src.cpcv.preprocessing import preprocess_fold
from src.cpcv.models import create_model, list_models
from src.cpcv.calibration import Calibrator
from src.cpcv.pipeline import run_cpcv_pipeline

# Post-CPCV
from src.post_cpcv.evaluation import analyze_results
from src.post_cpcv.symbolic_extraction import run_symbolic_extraction

# ── logging setup ─────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",)

print("✅ Libraries imported successfully!")

# 2. Data Engineering

## 2.1. BTC Daily OHLCV Data

In [ ]:
START_DATE, END_DATE = '2014-09-17', '2026-04-12'

df_raw = load_btc_daily('BTC-USD', start=START_DATE, end=END_DATE)
df_raw.head()

## 2.2. Labeling

### Daily Volatility

In [ ]:
# ── Step 1: Daily volatility ──────────────────────────────────────────
daily_vol = compute_daily_volatility(df_raw["Close"], span=50)
daily_vol.plot(title="Daily Volatility (EWMA span=50)", figsize=(12, 3))

### CUSUM filter

In [ ]:
# ── Step 2: CUSUM filter ──────────────────────────────────────────────
log_returns = np.log(df_raw["Close"] / df_raw["Close"].shift(1)).dropna()
CUSUM_MULT = 1.5

h = CUSUM_MULT * daily_vol.mean()
t_events = cusum_filter(log_returns, h)
print(f"CUSUM threshold: {h:.6f} | {len(t_events)} events detected")

In [ ]:
# ── CUSUM filter visualization (Jan–Mar 2026) ───────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                         gridspec_kw={"height_ratios": [1.2, 1]})

# zoom window
zoom_start, zoom_end = "2026-01-01", "2026-03-27"
mask = (log_returns.index >= zoom_start) & (log_returns.index <= zoom_end)
lr_zoom = log_returns.loc[mask]
events_zoom = t_events[(t_events >= zoom_start) & (t_events <= zoom_end)]

# ── Top panel: log returns + CUSUM events ────────────────────────────
ax1 = axes[0]
ax1.bar(lr_zoom.index, lr_zoom.values, width=0.8, alpha=0.6,
        color=["#2ecc71" if r > 0 else "#e74c3c" for r in lr_zoom.values],
        label="Log returns")
for ev in events_zoom:
    ax1.axvline(ev, color="#3498db", alpha=0.7, linewidth=1.2, linestyle="--")
# one dummy line for legend
ax1.axvline(events_zoom[0] if len(events_zoom) > 0 else lr_zoom.index[0],
            color="#3498db", alpha=0.7, linewidth=1.2, linestyle="--",
            label=f"CUSUM events ({len(events_zoom)})")
ax1.axhline(0, color="gray", linewidth=0.5)
ax1.set_ylabel("Log return")
ax1.set_title("CUSUM filter — Jan/Mar 2026", fontsize=13, fontweight="bold")
ax1.legend(loc="upper right", fontsize=9)

# ── Bottom panel: cumulative sums S+ and S- ─────────────────────────
ax2 = axes[1]
s_pos_series, s_neg_series = [], []
s_pos, s_neg = 0.0, 0.0
for t, r in lr_zoom.items():
    s_pos = max(0.0, s_pos + r)
    s_neg = min(0.0, s_neg + r)
    # reset if this is an event
    if t in events_zoom:
        s_pos_series.append(0.0)
        s_neg_series.append(0.0)
        s_pos, s_neg = 0.0, 0.0
    else:
        s_pos_series.append(s_pos)
        s_neg_series.append(s_neg)

ax2.fill_between(lr_zoom.index, s_pos_series, 0, alpha=0.3, color="#2ecc71", label="S⁺ (upward)")
ax2.fill_between(lr_zoom.index, s_neg_series, 0, alpha=0.3, color="#e74c3c", label="S⁻ (downward)")
ax2.axhline(h, color="#2ecc71", linewidth=1, linestyle=":", label=f"h = +{h:.4f}")
ax2.axhline(-h, color="#e74c3c", linewidth=1, linestyle=":", label=f"h = −{h:.4f}")
for ev in events_zoom:
    ax2.axvline(ev, color="#3498db", alpha=0.5, linewidth=1, linestyle="--")
ax2.set_ylabel("Cumulative sum")
ax2.set_xlabel("Date")
ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nCUSUM events in window: {len(events_zoom)}")
print(f"Total events (full sample): {len(t_events)}")

### Triple-Barrier Labeling

In [ ]:
# ── Step 3: Triple-barrier labeling ───────────────────────────────────
PT_SL_COMBO, NUM_DAYS = (1.5, 1.5), 10

bins = triple_barrier_labels(df_raw["Close"], t_events, trgt=daily_vol,
                             pt_sl=PT_SL_COMBO, num_days=NUM_DAYS, min_return=0.00)

holding_days = (bins["t1"] - bins.index).dt.days
print(f"\nHolding period: mean={holding_days.mean():.1f}, median={holding_days.median():.1f} days")
print(f"Hit vertical barrier: {(holding_days >= NUM_DAYS).mean():.1%}")

In [ ]:
# ── Triple-barrier labeling examples (Jan–Mar 2026) ─────────────────
zoom_start, zoom_end = "2026-01-01", "2026-03-27"
bins_zoom = bins[(bins.index >= zoom_start) & (bins.index <= zoom_end)].copy()

# classify each event by WHICH barrier was actually hit
# (not just the label, which only tells us the sign of the return)
classified = {"profit": [], "stop_loss": [], "vertical": []}

for t0 in bins_zoom.index:
    row = bins_zoom.loc[t0]
    t1 = row["t1"]
    label = int(row["bin"])
    hold = (t1 - t0).days

    p0 = df_raw["Close"].loc[t0]
    vol = daily_vol.loc[t0]
    upper = p0 * (1.0 + PT_SL_COMBO[0] * vol)
    lower = p0 * (1.0 - PT_SL_COMBO[1] * vol)

    # check the price path to see which barrier was touched first
    path = df_raw["Close"].loc[t0:t1]
    hit_upper = (path >= upper).any()
    hit_lower = (path <= lower).any()

    if hit_upper and label == 1:
        classified["profit"].append(t0)
    elif hit_lower and label == -1:
        classified["stop_loss"].append(t0)
    else:
        # neither horizontal barrier was hit → vertical barrier
        classified["vertical"].append(t0)

print("Barrier classification (Jan–Mar 2026):")
print(f"  Profit target hit:  {len(classified['profit'])}")
print(f"  Stop loss hit:      {len(classified['stop_loss'])}")
print(f"  Vertical barrier:   {len(classified['vertical'])}")

# pick one example per type, preferring 3+ day holds for visible price action
examples = {}
for key in ["profit", "stop_loss", "vertical"]:
    candidates = classified[key]
    if not candidates:
        continue
    # compute hold days
    holds = [(t0, (bins.loc[t0, "t1"] - t0).days) for t0 in candidates]
    # prefer 3+ day holds
    long_holds = [(t0, h) for t0, h in holds if h >= 3]
    if long_holds:
        examples[key] = long_holds[len(long_holds) // 2][0]
    else:
        examples[key] = holds[0][0]

plot_order = []
for key in ["profit", "stop_loss", "vertical"]:
    if key in examples:
        plot_order.append((key, examples[key]))

if len(plot_order) == 0:
    print("No TBL events found in Jan–Mar 2026. Try a wider window.")
else:
    n_examples = len(plot_order)
    fig, axes = plt.subplots(1, n_examples, figsize=(6 * n_examples, 5.5), sharey=False)
    if n_examples == 1:
        axes = [axes]

    panel_colors = {"profit": "#2ecc71", "stop_loss": "#e74c3c", "vertical": "#95a5a6"}
    panel_titles = {
        "profit": "Take profit",
        "stop_loss": "Stop loss",
        "vertical": "Vertical barrier",
    }

    for ax, (panel_key, t0) in zip(axes, plot_order):
        row = bins.loc[t0]
        t1 = row["t1"]
        label = int(row["bin"])
        ret = row["ret"]

        p0 = df_raw["Close"].loc[t0]
        vol = daily_vol.loc[t0]
        upper = p0 * (1.0 + PT_SL_COMBO[0] * vol)
        lower = p0 * (1.0 - PT_SL_COMBO[1] * vol)
        vert_date = t0 + pd.Timedelta(days=NUM_DAYS)

        # price path: start 1 day before entry, end 2 days after exit
        path_start = t0 - pd.Timedelta(days=1)
        path_end = t1 + pd.Timedelta(days=2)
        path = df_raw["Close"].loc[path_start:path_end]

        # ensure at least 5 days of price action are visible
        if len(path) < 5:
            path_end = t0 + pd.Timedelta(days=max(NUM_DAYS + 2, 7))
            path = df_raw["Close"].loc[path_start:path_end]

        # price path: gray before entry, dark during holding, gray after exit
        pre_entry = path.loc[:t0]
        during = path.loc[t0:t1]
        post_exit = path.loc[t1:]

        if len(pre_entry) > 0:
            ax.plot(pre_entry.index, pre_entry.values, color="#bdc3c7",
                    linewidth=1, zorder=2)
        ax.plot(during.index, during.values, color="#2c3e50",
                linewidth=2, zorder=3)
        if len(post_exit) > 1:
            ax.plot(post_exit.index, post_exit.values, color="#bdc3c7",
                    linewidth=1, zorder=2)

        # entry point
        ax.scatter([t0], [p0], color="#3498db", s=100, zorder=5,
                   edgecolors="white", linewidth=1.5, label="Entry")

        # exit point
        exit_price = df_raw["Close"].loc[t1] if t1 in df_raw["Close"].index else p0 * (1 + ret)
        ax.scatter([t1], [exit_price], color=panel_colors[panel_key], s=100,
                   zorder=5, marker="D", edgecolors="white", linewidth=1.5,
                   label=f"Exit ({(t1 - t0).days}d)")

        # barriers: only draw within the holding zone
        barrier_x = [t0, min(vert_date, path.index[-1])]
        ax.hlines(upper, barrier_x[0], barrier_x[1], color="#2ecc71",
                  linewidth=1.2, linestyle="--", alpha=0.8,
                  label=f"Profit: ${upper:,.0f}")
        ax.hlines(lower, barrier_x[0], barrier_x[1], color="#e74c3c",
                  linewidth=1.2, linestyle="--", alpha=0.8,
                  label=f"Stop: ${lower:,.0f}")

        # vertical barrier line
        if vert_date <= path.index[-1]:
            ax.axvline(vert_date, color="#95a5a6", linewidth=1.2,
                       linestyle=":", alpha=0.8, label=f"Vertical ({NUM_DAYS}d)")

        # shade the barrier zone
        zone = path.loc[t0:vert_date]
        if len(zone) > 0:
            ax.fill_between(zone.index, lower, upper, alpha=0.05, color="#3498db")

        # annotate the return
        ax.annotate(
            f"ret = {ret:+.2%}",
            xy=(t1, exit_price),
            xytext=(15, 15 if ret > 0 else -20),
            textcoords="offset points",
            fontsize=9, fontweight="bold",
            color=panel_colors[panel_key],
            arrowprops=dict(arrowstyle="->", color=panel_colors[panel_key], lw=0.8),
        )

        # formatting
        ax.set_title(
            f"{panel_titles[panel_key]}\n"
            f"{t0.strftime('%b %d')} → {t1.strftime('%b %d')} ({(t1 - t0).days} days)",
            fontsize=11, fontweight="bold", color=panel_colors[panel_key],
        )
        ax.legend(fontsize=7, loc="best", framealpha=0.9)
        ax.tick_params(axis="x", rotation=30, labelsize=8)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
        ax.grid(alpha=0.15)
        ax.set_xlabel("")

    fig.suptitle(
        f"Triple-barrier labeling — pt_sl={PT_SL_COMBO}, num_days={NUM_DAYS}",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.show()

    # summary stats
    print(f"\nTBL events in Jan–Mar 2026: {len(bins_zoom)}")
    print(f"  Profit target hit:  {len(classified['profit'])}")
    print(f"  Stop loss hit:      {len(classified['stop_loss'])}")
    print(f"  Vertical barrier:   {len(classified['vertical'])}")
    holding = (bins_zoom["t1"] - bins_zoom.index).dt.days
    print(f"  Avg holding period: {holding.mean():.1f} days")

### Drop rare labels

In [ ]:
# ── Step 4: Drop rare labels ─────────────────────────────────────────
bins = drop_rare_labels(bins, min_pct=0.05)
print(f"{len(bins)} labels | classes: {bins['bin'].value_counts().to_dict()}")
bins.head(10)

### Label Distribution

In [ ]:
# ── Class distribution ────────────────────────────────────────────
counts = bins["bin"].value_counts().sort_index()

label_map = {-1: "-1 (Down)", 1: "1 (Up)"}
labels = [label_map.get(x, str(x)) for x in counts.index]
colors = ["#800000", "#0b6623"][:len(counts)]  # red, green

plt.figure(figsize=(8, 5))
patches, texts, autotexts = plt.pie(
    counts, labels=labels, autopct="%1.1f%%",
    startangle=140, colors=colors, pctdistance=0.85
)

centre_circle = plt.Circle((0, 0), 0.25, fc="white")
plt.gcf().gca().add_artist(centre_circle)

plt.legend(patches, labels, title="Price Movement",
           loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
plt.title("Label Distribution", fontsize=16, fontweight="bold")
plt.axis("equal")
plt.tight_layout()
plt.show()

print(f"Total samples: {len(bins)}")
for cls, n in counts.items():
    print(f"  Class {label_map.get(cls, cls)}: {n} ({n/len(bins)*100:.1f}%)")

## 2.3. Sample Weights

In [ ]:
# ── Step 5: Sample weights ────────────────────────────────────────────
sample_w = compute_sample_weights(bins, df_raw.index, time_decay_factor=0.5,
                                  weight_cap_quantile=0.99) # weight_cap_quantile = 1.00 to disable capping
sample_w.describe()

In [ ]:
# ── Visualize sample weights ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(sample_w, bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(sample_w.mean(), color="red", ls="--", label=f"mean={sample_w.mean():.2f}")
axes[0].set_title("Sample Weight Distribution")
axes[0].set_xlabel("Weight")
axes[0].legend()

axes[1].plot(sample_w.index, sample_w.values, lw=0.6)
axes[1].set_title("Sample Weights Over Time")
axes[1].set_ylabel("Weight")

plt.tight_layout()
plt.show()

In [ ]:
# ── Heavy-weight events (diagnostic) ─────────────────────────────────
heavy_threshold = sample_w.quantile(0.99)
heavy = sample_w[sample_w >= heavy_threshold].sort_values(ascending=False)
heavy_df = pd.DataFrame({
    "weight": heavy,
    "label": bins.loc[heavy.index, "bin"],
    "return": bins.loc[heavy.index, "ret"],
    "holding_days": (bins.loc[heavy.index, "t1"] - heavy.index).dt.days,
})
print(f"Events at or above 99th percentile ({heavy_threshold:.2f}): {len(heavy_df)}\n")
print(heavy_df.to_string())

## 2.4. Feature Engineering

In [ ]:
# ── Technical Analysis features ──────────────────────────────
ta_features = compute_ta_features(df_raw)
print(f"TA features: {list(ta_features.columns)}")
print(f"Shape: {ta_features.shape}")
print(f"NaN rows (any): {ta_features.isna().any(axis=1).sum()}")
ta_features.head(30)

In [ ]:
# ── External features (macro + crypto) ───────────────────────────────
external = build_external_features(df_raw)
print(f"External features: {list(external.columns)}")
print(f"Shape: {external.shape}")
external.head(10)

In [ ]:
# ── Mathematical features (AFML Part 4) ─────────────────────
# full run (~45 min, cached after first run)
math_features = compute_math_features(df_raw, which="all")
# quick run (seconds)
#math_features = compute_math_features(df_raw, which=["shannon_entropy", "lz_complexity", "hurst", "variance_ratio", "jarque_bera", "gaussian_entropy"])
print(f"Math features: {list(math_features.columns)}")
print(f"Shape: {math_features.shape}")
print(f"NaN rows (any): {math_features.isna().any(axis=1).sum()}")
math_features.head(10)

In [ ]:
# ── Assemble feature matrix & Log transforms ─────────────────
feature_matrix = pd.concat([ta_features, math_features, external], axis=1)
feature_matrix = apply_log_transforms(feature_matrix)

# drop columns with >50% NaN
nan_pct = feature_matrix.isna().mean()
high_nan = nan_pct[nan_pct > 0.50].index.tolist()
if high_nan:
    feature_matrix = feature_matrix.drop(columns=high_nan)
    print(f"⚠ Dropped {len(high_nan)} column(s) with >50% NaN: {high_nan}")

print(f"Final feature matrix: {feature_matrix.shape[1]} features, {feature_matrix.shape[0]} rows")
print(f"Columns: {list(feature_matrix.columns)}")
print(f"NaN rows (any): {feature_matrix.isna().any(axis=1).sum()}")
feature_matrix.describe()

## 2.5. Exploratory Data Analysis

### Feature Distributions

In [ ]:
# ── Feature distributions ────────────────────────────────────────
n_cols = 4
feat_cols = feature_matrix.columns.tolist()
n_rows = int(np.ceil(len(feat_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
axes = axes.flatten()

flagged = []
for i, col in enumerate(feat_cols):
    data = feature_matrix[col].dropna()
    axes[i].hist(data, bins=50, edgecolor="black", alpha=0.7)
    kurt = data.kurtosis()
    title_suffix = f" [kurt={kurt:.1f}]" if kurt > 10 else ""
    axes[i].set_title(f"{col}{title_suffix}", fontsize=9)
    if kurt > 10:
        axes[i].set_title(f"{col}{title_suffix}", fontsize=9, color="red")
        flagged.append((col, kurt))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

if flagged:
    print("⚠ Features with kurtosis > 10 (may saturate KAN spline ranges):")
    for col, k in flagged:
        print(f"  {col}: kurtosis = {k:.1f}")
else:
    print("✓ No features with kurtosis > 10.")

### Feature Correlation

In [ ]:
corr = feature_matrix.dropna().corr()

n = len(corr.columns)
fig, ax = plt.subplots(figsize=(max(20, n * 0.55), max(18, n * 0.5)))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(corr.columns, fontsize=10)
fig.colorbar(im, ax=ax, shrink=0.8)

# annotate cells
for i in range(n):
    for j in range(n):
        val = corr.iloc[i, j]
        color = "white" if abs(val) > 0.6 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=10, color=color)

ax.set_title("Feature Correlation Matrix", fontsize=18, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

# flag high-correlation pairs
corr_threshold = 0.9
high_corr = []
for i in range(n):
    for j in range(i + 1, n):
        r = corr.iloc[i, j]
        if abs(r) > corr_threshold:
            high_corr.append((corr.columns[i], corr.columns[j], r))

if high_corr:
    print(f"⚠ {len(high_corr)} pair(s) with |correlation| > {corr_threshold}:")
    for c1, c2, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {c1} ↔ {c2}: {r:.3f}")
else:
    print(f"✓ No feature pairs with |correlation| > {corr_threshold}.")

### Stationarity ADF test

In [ ]:
# ── Stationarity (ADF test per feature) ──────────────────────────
adf_results = []
for col in feature_matrix.columns:
    series = feature_matrix[col].dropna()
    if len(series) < 50:
        continue
    stat, pval, *_ = adfuller(series, maxlag=14, autolag="AIC")
    adf_results.append({"feature": col, "adf_stat": stat, "p_value": pval})

adf_df = pd.DataFrame(adf_results).set_index("feature")
adf_df["stationary_5pct"] = adf_df["p_value"] < 0.05

print("ADF test results (H₀: unit root exists):\n")
for _, row in adf_df.iterrows():
    status = "✓ stationary" if row["stationary_5pct"] else "✗ non-stationary"
    print(f"  {row.name:20s}  p={row['p_value']:.4f}  {status}")

non_stat = adf_df[~adf_df["stationary_5pct"]]
if len(non_stat):
    print(f"\n⚠ {len(non_stat)} non-stationary feature(s) at 5%: {list(non_stat.index)}")
    print("  Expected for cumulative features (OBV). Will be handled by FFD in the CV loop.")
else:
    print("\n✓ All features stationary at 5%.")

# auto-detect non-stationary features for FFD
COLUMNS_TO_FFD = ['atr']
print(f"\nColumns for FFD: {COLUMNS_TO_FFD}")

### Mutual Infomation - Features VS Label 

In [ ]:
# ── Feature vs label (mutual information) ────────────────────────
# align features to labeled events
aligned = feature_matrix.loc[feature_matrix.index.isin(bins.index)].copy()
aligned = aligned.loc[aligned.index.isin(bins.index)]
y_aligned = bins.loc[aligned.index, "bin"]

# drop rows with any NaN for MI computation
mask = aligned.notna().all(axis=1)
X_clean = aligned.loc[mask]
y_clean = y_aligned.loc[mask]

mi = mutual_info_classif(X_clean, y_clean, random_state=42, n_neighbors=5)
mi_series = pd.Series(mi, index=X_clean.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 10))
mi_series.plot.barh(ax=ax, edgecolor="black", alpha=0.7)
ax.set_xlabel("Mutual Information (nats)")
ax.set_title("Feature–Label Mutual Information")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

zero_mi = mi_series[mi_series < 1e-6]
if len(zero_mi):
    print(f"⚠ {len(zero_mi)} feature(s) with near-zero MI (removal candidates): {list(zero_mi.index)}")
else:
    print("✓ All features show non-zero mutual information with the label.")

## 2.6. Data Alignment

In [ ]:
# ── Alignment ────────────────────────────────────────────────
X, y, w, t1 = align_for_cv(feature_matrix, bins, sample_w)
validate_alignment(X, y, w, t1)
X.head()

# 3. Cross-Validation Framework

## 3.1. CPCV Split Generation

In [ ]:
# ── CPCV: Split generation ────────────────────────────────────────────
split_info = get_split_info(X, t1)
splits = generate_cpcv_splits(X, t1)
n_paths, path_map = build_path_matrix()

## 3.2. CPCV EDA

### Partition Overview

In [ ]:
# ── Group partition overview ──────────────────────────────────────
T = len(X)
n_groups = 6
base_size = T // n_groups
group_bounds = []
for g in range(n_groups):
    start = g * base_size
    end = (g + 1) * base_size if g < n_groups - 1 else T
    group_bounds.append((start, end))

colors_g = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db", "#9b59b6"]

fig, ax = plt.subplots(figsize=(14, 5))

# foreground: BTC close price (restricted to aligned range)
ax.plot(X.index, df_raw.loc[X.index, "Close"], color="black", linewidth=0.8)
ax.set_yscale("log")

# background: colored regions per group
for g, (start, end) in enumerate(group_bounds):
    ax.axvspan(X.index[start], X.index[end - 1],
               color=colors_g[g], alpha=0.15)

# place group labels inside the plot area
ymin, ymax = ax.get_ylim()
for g, (start, end) in enumerate(group_bounds):
    mid_idx = (start + end) // 2
    ax.text(X.index[mid_idx], ymin * 1.5, f"G{g}",
            ha="center", va="bottom", fontsize=11, fontweight="bold",
            color=colors_g[g])

ax.set_ylabel("BTC Close (USD, log scale)")
ax.set_xlabel("")
ax.set_title("BTC Price with CPCV Group Partitions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Train / Test Splits

In [ ]:
# ── Train/test timelines (3 representative splits) ───────────────
# Split 0:  contiguous test (G0, G1)
# Split 6:  one-gap test   (G1, G3)
# Split 14: contiguous tail (G4, G5)
demo_splits = [0, 6, 14]
all_combos = list(__import__("itertools").combinations(range(n_groups), 2))

fig, axes = plt.subplots(len(demo_splits), 1, figsize=(14, 1.8 * len(demo_splits)), sharex=True)

for ax, split_id in zip(axes, demo_splits):
    train_idx, test_idx = splits[split_id]
    test_groups = all_combos[split_id]

    # plot train/test points
    ax.scatter(X.index[train_idx], np.zeros(len(train_idx)),
               c="steelblue", s=2, label="Train", zorder=2)
    ax.scatter(X.index[test_idx], np.zeros(len(test_idx)),
               c="crimson", s=2, label="Test", zorder=3)

    # shade test groups
    for g in test_groups:
        g_start, g_end = group_bounds[g]
        ax.axvspan(X.index[g_start], X.index[g_end - 1],
                   color="crimson", alpha=0.08, zorder=1)

    # mark group boundaries
    for g in range(n_groups):
        g_start, _ = group_bounds[g]
        ax.axvline(X.index[g_start], color="gray", ls="--", lw=0.5, alpha=0.4)

    ax.set_yticks([])
    ax.set_title(f"Split {split_id}: test groups {test_groups} — "
                 f"train={len(train_idx)}, test={len(test_idx)}",
                 fontsize=9, fontweight="bold")
    ax.legend(loc="upper right", fontsize=7, markerscale=3)

plt.xlabel("Date")
plt.suptitle("CPCV Train/Test Timelines (3 Representative Splits)",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### Purging & Embargo Verification

In [ ]:
# ── Purging & embargo detail ─────────────────────────────────────
print("Purging & Embargo Verification (boundary observations)\n")

for split_id in demo_splits:
    train_idx, test_idx = splits[split_id]
    test_groups = all_combos[split_id]

    print(f"{'─'*60}")
    print(f"Split {split_id} — test groups {test_groups}")
    print(f"{'─'*60}")

    for g in test_groups:
        g_start, g_end = group_bounds[g]
        t_test_start = X.index[g_start]
        t_test_end = X.index[g_end - 1]

        # last 3 training obs BEFORE this test group
        train_before = sorted([i for i in train_idx if i < g_start])
        # first 3 training obs AFTER this test group
        train_after = sorted([i for i in train_idx if i >= g_end])

        print(f"\n  Test G{g}: [{t_test_start.date()} → {t_test_end.date()}]")

        if train_before:
            last_train_before = train_before[-1]
            purge_gap = g_start - last_train_before - 1
            print(f"  ┌ Pre-test gap (purged): {purge_gap} obs removed")
            for i in train_before[-3:]:
                safe = t1.iloc[i] < t_test_start
                print(f"  │   idx={i:>4d} ({X.index[i].date()})  "
                      f"t1={t1.iloc[i].date()}  {'✓' if safe else '✗ OVERLAP'}")

        if train_after:
            first_train_after = train_after[0]
            embargo_gap = first_train_after - g_end
            print(f"  └ Post-test gap (embargo): {embargo_gap} obs removed")
            for i in train_after[:3]:
                print(f"      idx={i:>4d} ({X.index[i].date()})")

    print()

### Leakage Verification

In [ ]:
# ── Leakage audit (all splits) ────────────────────────────────────
n_leaks_total = 0
audit_results = []

for i, (train_idx, test_idx) in enumerate(splits):
    test_groups = all_combos[i]
    leak_count = 0

    for g in test_groups:
        g_start, g_end = group_bounds[g]
        t_test_start = X.index[g_start]
        t_test_end = X.index[g_end - 1]

        train_t1 = t1.iloc[train_idx]
        train_times = X.index[train_idx]

        leaks = train_t1[
            (train_times < t_test_start) &
            (train_t1 >= t_test_start) &
            (train_t1 <= t_test_end)
        ]
        leak_count += len(leaks)

    n_leaks_total += leak_count
    audit_results.append({
        "split": i,
        "test_groups": test_groups,
        "train": len(train_idx),
        "test": len(test_idx),
        "leaks": leak_count,
    })

# summary table
audit_results_df = pd.DataFrame(audit_results)
audit_results_df["status"] = audit_results_df["leaks"].apply(lambda x: "✓" if x == 0 else "✗")

print("CPCV Leakage Audit")
print("=" * 65)
for _, row in audit_results_df.iterrows():
    print(f"  Split {row['split']:2d} | groups {str(row['test_groups']):>7s} | "
          f"train={row['train']:>4d}  test={row['test']:>4d} | "
          f"leaks={row['leaks']:>2d} {row['status']}")

print("=" * 65)
if n_leaks_total == 0:
    print(f"✓ All {len(splits)} splits passed. Zero leakage detected.")
    print(f"  {split_info['n_paths']} backtest paths available for Sharpe ratio distribution.")
else:
    print(f"✗ {n_leaks_total} total leaking observations detected.")

## 3.3. Preprocessing Demo

In [ ]:
# ── Preprocessing: demo on split 0 ───────────────────────────────────
train_idx, test_idx = splits[0]

# extract labels, weights, t1 for this fold
y_train_raw = ((y + 1) // 2).astype(int).iloc[train_idx]
w_train_raw = w.iloc[train_idx]
t1_train_raw = t1.iloc[train_idx]

X_tr, X_te, selected, info = preprocess_fold(
    X, train_idx, test_idx, y_train_raw, w_train_raw, t1_train_raw,
    ffd_columns=COLUMNS_TO_FFD, top_k_frac=0.4)

print(f"\nTrain: {X_tr.shape}, Test: {X_te.shape}")
print(f"FFD d* values: {info['ffd']}")
print(f"Selected features: {info['selected_features']}")

In [ ]:
# ── Preprocessing: inspect outputs ───────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(17, 10))

# scaled feature distributions (training)
X_tr.boxplot(ax=axes[0], rot=45, fontsize=7)
axes[0].set_title("Scaled Training Features")

# compare train vs test distributions (check for drift)
X_tr.mean().plot(ax=axes[1], label="Train mean", marker="o", ms=4)
X_te.mean().plot(ax=axes[1], label="Test mean", marker="x", ms=4)
axes[1].set_title("Train vs Test Feature Means")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45, labelsize=7)

plt.tight_layout()
plt.show()

## 3.4. Modeling Demo

#### Split 0

In [ ]:
# ── Smoke test: Logistic Regression on split 0 ───────────────────────
# X_tr, X_te come from the preprocessing demo cell (split 0, post-preprocess)
# map labels {-1, +1} → {0, 1}
y_tr_m = ((y.loc[X_tr.index] + 1) // 2).astype(int)
y_te_m = ((y.loc[X_te.index] + 1) // 2).astype(int)
w_tr_m = w.loc[X_tr.index]

# apply feature selection
X_tr_sel = X_tr[selected]
X_te_sel = X_te[selected]

# 80/20 chronological split for calibration
cal_boundary = int(len(X_tr_sel) * 0.8)
X_model, X_cal = X_tr_sel.iloc[:cal_boundary], X_tr_sel.iloc[cal_boundary:]
y_model, y_cal = y_tr_m.iloc[:cal_boundary], y_tr_m.iloc[cal_boundary:]
w_model = w_tr_m.iloc[:cal_boundary]

# fit
model = create_model("logistic", n_features=len(selected), seed=42)
model.fit(X_model, y_model, sample_weight=w_model)

# calibrate
cal = Calibrator()
cal.fit(model, X_cal, y_cal)
print(cal)

# predict
raw_logits = model.predict_logits(X_te_sel)
cal_proba = cal.calibrate(raw_logits)
y_pred = np.argmax(cal_proba, axis=1)
y_true = y_te_m.values

# ── metrics ───────────────────────────────────────────────────────────
print(f"\n{'─'*40}")
print(f"Logistic Regression — Split 0 Results")
print(f"{'─'*40}")

f1 = f1_score(y_true, y_pred, average="macro")
roc_auc = roc_auc_score(y_true, cal_proba[:, 1])
print(f"  F1 (macro):  {f1:.4f}")
print(f"  ROC-AUC:     {roc_auc:.4f}")
print(f"  Selected:    {selected}")
print()
print(classification_report(y_true, y_pred, target_names=["Down (0)", "Up (1)"]))

# ── plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=["Down", "Up"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(
    y_true, cal_proba[:, 1], ax=axes[1], name="Logistic"
)
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8, label="Random (AUC=0.50)")
axes[1].legend()
axes[1].set_title("ROC Curve")

plt.tight_layout()
plt.show()

#### Split 14

In [ ]:
# ── Smoke test: Logistic Regression on last split ────────────────────
# use last split
last_split_idx = len(splits) - 1
train_idx, test_idx = splits[last_split_idx]

y_train_raw = ((y + 1) // 2).astype(int).iloc[train_idx]
w_train_raw = w.iloc[train_idx]
t1_train_raw = t1.iloc[train_idx]

X_tr_last, X_te_last, selected_last, info_last = preprocess_fold(
    X, train_idx, test_idx, y_train_raw, w_train_raw, t1_train_raw,
    ffd_columns=COLUMNS_TO_FFD, top_k_frac=0.4)

print(f"Available models: {list_models()}\n")

y_tr_m = ((y.loc[X_tr_last.index] + 1) // 2).astype(int)
y_te_m = ((y.loc[X_te_last.index] + 1) // 2).astype(int)
w_tr_m = w.loc[X_tr_last.index]

# apply feature selection
X_tr_sel = X_tr_last[selected_last]
X_te_sel = X_te_last[selected_last]

# 80/20 chronological split for calibration
cal_boundary = int(len(X_tr_sel) * 0.8)
X_model, X_cal = X_tr_sel.iloc[:cal_boundary], X_tr_sel.iloc[cal_boundary:]
y_model, y_cal = y_tr_m.iloc[:cal_boundary], y_tr_m.iloc[cal_boundary:]
w_model = w_tr_m.iloc[:cal_boundary]

# fit
model = create_model("logistic", n_features=len(selected_last), seed=42)
model.fit(X_model, y_model, sample_weight=w_model)

# calibrate
cal = Calibrator()
cal.fit(model, X_cal, y_cal)
print(cal)

# predict
raw_logits = model.predict_logits(X_te_sel)
cal_proba = cal.calibrate(raw_logits)
y_pred = np.argmax(cal_proba, axis=1)
y_true = y_te_m.values

# ── metrics ───────────────────────────────────────────────────────────
print(f"\n{'─'*40}")
print(f"Logistic Regression — Split {last_split_idx} Results")
print(f"{'─'*40}")

f1 = f1_score(y_true, y_pred, average="macro")
roc_auc = roc_auc_score(y_true, cal_proba[:, 1])
print(f"  F1 (macro):  {f1:.4f}")
print(f"  ROC-AUC:     {roc_auc:.4f}")
print(f"  Selected:    {selected_last}")
print()
print(classification_report(y_true, y_pred, target_names=["Down (0)", "Up (1)"]))

# ── plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=["Down", "Up"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title(f"Confusion Matrix (Split {last_split_idx})")

RocCurveDisplay.from_predictions(
    y_true, cal_proba[:, 1], ax=axes[1], name="Logistic"
)
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8, label="Random (AUC=0.50)")
axes[1].legend()
axes[1].set_title(f"ROC Curve (Split {last_split_idx})")

plt.tight_layout()
plt.show()

# 4. Model Training

## 4.1. Benchmark Model

In [ ]:
# ── Econometric baseline (AR Logistic, 3 seeds, ~5 secs) ──────────────
ar_logistic_results = run_cpcv_pipeline(X, y, w, t1, bins_ret=bins["ret"], 
                      models=["ar_logistic"], n_seeds=3, ffd_columns=COLUMNS_TO_FFD,)

## 4.2. Logistic Regression

In [ ]:
TOP_K_FRAC = 0.4

In [ ]:
# ── Logistic Regression (3 seeds, ~8 min) ────────────────────────────────────
lr_results = run_cpcv_pipeline(X, y, w, t1, bins_ret=bins["ret"],
             models=["logistic"], n_seeds=3, ffd_columns=COLUMNS_TO_FFD,
             top_k_frac=TOP_K_FRAC, tune=True, tune_models=["logistic"],
             n_trials=30)

## 4.3. Ensemble Models

In [ ]:
# ── Ensemble Models (3 seeds, ~29 min) ────────────────────────────────────
ensembles_results = run_cpcv_pipeline(X, y, w, t1, bins_ret=bins["ret"],
                    models=["random_forest", "xgboost"], n_seeds=3, ffd_columns=COLUMNS_TO_FFD,
                    top_k_frac=TOP_K_FRAC, tune=True, tune_models=["random_forest", "xgboost"],
                    n_trials=30)

## 4.4. LSTM

In [ ]:
# ── LSTM (2 seed, ~36 min) ────────────────────────────────────
lstm_results = run_cpcv_pipeline(X, y, w, t1, bins_ret=bins["ret"],
               models=["lstm"], n_seeds=2, ffd_columns=COLUMNS_TO_FFD,
               top_k_frac=TOP_K_FRAC, tune=True, tune_models=["lstm"],
               n_trials=25)

## 4.5. KAN

In [ ]:
# ── KAN (2 seed, ~25 min) ────────────────────────────────────
kan_results = run_cpcv_pipeline(X, y, w, t1, bins_ret=bins["ret"],
              models=["kan"], n_seeds=2, ffd_columns=COLUMNS_TO_FFD,
              top_k_frac=TOP_K_FRAC, tune=True, tune_models=["kan"],
              n_trials=25)

## 4.6. Merge Results

In [ ]:
# ── Merge results ────────────────────────────────────────────────────
all_predictions = {
    **ar_logistic_results["predictions"],
    **lr_results["predictions"],
    **ensembles_results["predictions"],
    **lstm_results["predictions"],
    **kan_results["predictions"],
}

results = {
    "predictions": all_predictions,
    "split_info": lr_results["split_info"],
    "path_map": lr_results["path_map"],
    "n_paths": lr_results["n_paths"],
    "n_splits": lr_results["n_splits"],
    "models": ["ar_logistic", "logistic", "random_forest", "xgboost", "lstm", "kan"],
    "n_seeds": 3,
}

print(f"Merged: {len(all_predictions)} total prediction entries")
print(f"Models: {results['models']}")

# 5. Post-CPCV Evaluation

## 5.1. Model Comparison

In [ ]:
analysis = analyze_results(results)

## 5.2. Confusion Matrices

In [ ]:
# ── Confusion matrices (seed=0, averaged across splits) ──────────────

model_names = results["models"]
n_cols = 3
n_rows = int(np.ceil(len(model_names) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, model_name in enumerate(model_names):
    # aggregate y_true and y_pred across all splits (seed=0)
    all_true, all_pred = [], []
    for split_idx in range(results["n_splits"]):
        key = (model_name, split_idx, 0)
        if key not in results["predictions"]:
            continue
        pred = results["predictions"][key]
        all_true.append(pred["y_true"])
        all_pred.append(pred["y_pred"])

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Down", "Up"])
    disp.plot(ax=axes[i], cmap="Blues", colorbar=False)
    
    acc = np.mean(y_true == y_pred)
    axes[i].set_title(f"{model_name}\n(acc={acc:.3f}, n={len(y_true)})", fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Confusion Matrices (aggregated across all splits, seed=0)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 5.3. Feature Sensibility

In [ ]:
# ── Feature selection frequency across folds ─────────────────────────
feat_stab = analysis["feature_stability"]
freq = feat_stab["feature_frequency"].sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(6, len(freq) * 0.3)))
colors = ["#0b6623" if f > 0.8 else "#4a90d9" if f > 0.5 else "#999999" for f in freq.values]
ax.barh(freq.index, freq.values, color=colors)
ax.axvline(0.8, color="red", linestyle="--", alpha=0.7, label="80% threshold")
ax.set_xlabel("Selection Frequency (across 15 folds)")
ax.set_title("Feature Selection Stability")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nStable features (>80%): {feat_stab['stable_features']}")

## 5.4. FFD Stability

In [ ]:
# ── FFD d* values across folds ───────────────────────────────────────
ffd_stab = analysis["ffd_stability"]
for col, vals in ffd_stab["d_star_by_column"].items():
    print(f"{col}: mean d*={ffd_stab['mean_d_star'][col]:.3f}, "
          f"std={ffd_stab['std_d_star'][col]:.3f}, values={vals}")

## 5.5. PBO & DSR Summary

In [ ]:
# ── Probability of Backtest Overfitting ──────────────────────────────
print(f"PBO: {analysis['pbo']:.4f}")
if analysis["pbo"] < 0.3:
    print("  Model selection appears robust (PBO < 0.3).")
elif analysis["pbo"] > 0.5:
    print("  Warning: in-sample winner tends to underperform OOS (PBO > 0.5).")
else:
    print("  Moderate overfitting risk (0.3 < PBO < 0.5).")

# DSR per model
print("\nDeflated Sharpe Ratios:")
for s in analysis["all_summaries"]:
    dsr_flag = " ✓" if s["dsr"] > 0.95 else ""
    print(f"  {s['model_name']:>20s}: DSR={s['dsr']:.4f}{dsr_flag}")

## 5.6. Equity Curves

In [ ]:
# ── Equity curves per model (path 0, seed 0) ─────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
for model_name in results["models"]:
    path_data = analysis["path_results"][model_name]
    if 0 in path_data and len(path_data[0]["returns"]) > 0:
        equity = (1 + path_data[0]["returns"]).cumprod()
        ax.plot(equity.index, equity.values, label=model_name, linewidth=0.9)

ax.axhline(1.0, color="black", linestyle="--", alpha=0.3)
ax.set_ylabel("Cumulative Equity")
ax.set_title("CPCV Path 0: Equity Curves (seed=0)", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 5.7. Sharpe Distribution

In [ ]:
# ── Sharpe ratio distribution across 5 CPCV paths ───────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sharpe_data = []
labels = []
for i, model_name in enumerate(results["models"]):
    sharpes = analysis["path_sharpes"][i, :]
    sharpe_data.append(sharpes)
    labels.append(model_name)

ax.boxplot(sharpe_data, labels=labels, patch_artist=True)
ax.axhline(0, color="red", linestyle="--", alpha=0.5, label="Sharpe = 0")
ax.set_ylabel("Annualized Sharpe Ratio")
ax.set_title("Sharpe Ratio Distribution Across 5 CPCV Paths", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

# 6. Symbolic Extraction

In [ ]:
symbolic = run_symbolic_extraction(cpcv_results=results, X=X, y=y, w=w, t1=t1,
                                   n_top_features=5, use_multkan=False,
                                   fold_selection="last")

## 6.1. Formula VS KAN

In [ ]:
# ── Interpretable formula ────────────────────────────────────────────
print("Decision function:")
print(f"  {symbolic['decision_function']}")
print(f"\nP(up) = {symbolic['p_up_formula']}")
print(f"\nSurviving features: {symbolic['surviving_features']}")
print(f"Pre-symbolic accuracy:  {symbolic['pre_symbolic_accuracy']:.4f}")
print(f"Post-symbolic accuracy: {symbolic['post_symbolic_accuracy']:.4f}")

In [ ]:
# ── Feature sensitivity analysis ─────────────────────────────────────
decision_expr = symbolic["sympy_objects"]["decision"]
if decision_expr is not None and decision_expr != "extraction_failed":
    print("Partial derivatives (feature sensitivity):\n")
    for feat in symbolic["surviving_features"]:
        sensitivity = sympy.diff(decision_expr, sympy.Symbol(feat))
        print(f"  d(decision)/d({feat}) = {sensitivity}")